# DACON 딥보이스 탐지 - SONICS 기반 4개 소스 풀 × 25,000 Dynamic Mix + RawBoost/AASIST

이 노트북은 MusicGen을 실행하지 않습니다. 기존 공개 데이터셋만 코드로 내려받아 다음 네 소스 풀을 구성합니다.

1. Real Voice: Kaggle Deepfake Audio Dataset의 `real/`
2. Fake Voice: 같은 Kaggle 데이터셋의 `fake/`
3. Real Music: FMA medium의 small+medium subset
4. Fake Music: SONICS의 Suno/Udio 가짜 곡

각 풀에서 정확히 6,250개를 선택해 source reference 25,000개를 만들고, train/validation/audit용 동적 믹싱 recipe도 정확히 25,000개 생성합니다. 기본 모델은 공식 AASIST + RawBoost이며 DACON 공식 5개 확률, EER/ROC-AUC/ADS/CPS/Score와 `submit.zip` 규격을 사용합니다.


## 설계 원칙과 라벨

모델 출력 순서는 아래와 같습니다.

`FILE_FAKE, VOICE_FAKE, MUSIC_FAKE, VOICE_PRESENT, MUSIC_PRESENT`

- FAKE는 공식 정의대로 양성 클래스 `1`, REAL은 `0`입니다.
- 파일은 음성 또는 음악 중 하나라도 FAKE이면 FAKE입니다.
- 성분이 없는 샘플의 `VOICE_FAKE` 또는 `MUSIC_FAKE` loss는 mask 처리합니다. 공식 EER도 해당 성분이 존재하는 샘플에서만 계산됩니다.
- FMA에는 보컬이 포함될 수 있습니다. 보컬은 대회에서 음성 성분이므로 PANNs AudioSet tagger로 보컬 가능성을 스크리닝해 `VOICE_PRESENT=1, VOICE_FAKE=0`을 추가합니다.
- SONICS `no_vocal` 메타데이터를 이용해 보컬이 포함된 가짜 곡은 `VOICE_PRESENT=1`, `VOICE_FAKE=1`로 함께 라벨링합니다.
- 소스 파일은 train/validation/audit 사이에 절대 공유하지 않습니다.
- 평가 파일은 각각 독립적으로 추론하며 다른 평가 파일의 예측이나 통계를 사용하지 않습니다.

주의: FMA 오디오는 트랙별 아티스트 선택 라이선스를 따릅니다. metadata의 license를 보존하고 NoDerivatives 계열은 기본 제외합니다. SONICS 데이터셋은 CC BY-NC 4.0이므로 연구 목적과 대회 규칙을 확인하고 source manifest에 출처·라이선스를 보존하세요.


## 0. Colab 설치

GPU 런타임(T4/L4/A100)을 선택합니다. SONICS는 생성 없이 Hugging Face에서 두 개의 공식 ZIP을 내려받습니다. 학습 checkpoint와 manifest는 Drive에 저장되어 세션 재시작 후 이어집니다.


In [ ]:
import subprocess
import sys

packages = [
    "kaggle>=1.7", "transformers>=4.57,<5", "accelerate>=1.9", "huggingface_hub>=0.34",
    "librosa==0.10.2.post1", "soundfile>=0.12", "scikit-learn>=1.4",
    "panns-inference==0.1.1", "seaborn>=0.13", "pandas>=2.0",
    "scipy>=1.11", "einops>=0.8", "tqdm>=4.66",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
print("설치 완료. import 오류가 남으면 런타임을 한 번 재시작하세요.")


In [ ]:
from __future__ import annotations

import gc
import hashlib
import importlib.util
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import time
import warnings
import zipfile
from pathlib import Path
from types import SimpleNamespace

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from scipy.optimize import minimize
from sklearn.metrics import roc_auc_score, roc_curve
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

from google.colab import drive
drive.mount("/content/drive")
warnings.filterwarnings("ignore", category=FutureWarning)

CFG = SimpleNamespace(
    seed=42,
    sample_rate=16_000,
    # 공식 AASIST 입력 길이(64,600 samples @ 16 kHz)에 맞춘다.
    clip_seconds=4.0375,
    clip_samples=64_600,
    source_per_pool=6_250,
    source_split_counts={"train": 5_000, "validation": 625, "audit": 625},
    recipe_counts={"train": 20_000, "validation": 2_500, "audit": 2_500},
    panns_seconds=10,
    panns_batch=8,
    num_workers=4,
)
assert sum(CFG.source_split_counts.values()) == CFG.source_per_pool
assert sum(CFG.recipe_counts.values()) == 25_000

DACON_PROBABILITY_COLUMNS = [
    "FILE_FAKE_PROB", "VOICE_FAKE_PROB", "MUSIC_FAKE_PROB",
    "VOICE_PRESENT_PROB", "MUSIC_PRESENT_PROB",
]
DACON_TRUTH_COLUMNS = [
    "FILE_FAKE", "VOICE_FAKE", "MUSIC_FAKE", "VOICE_PRESENT", "MUSIC_PRESENT",
]
HEAD_WEIGHTS = torch.tensor([0.45, 0.18, 0.27, 0.05, 0.05], dtype=torch.float32)
AUDIO_SUFFIXES = {".wav", ".mp3", ".flac", ".ogg", ".m4a", ".aac", ".amr"}

DRIVE_ROOT = Path("/content/drive/MyDrive/deepvoice_dynamic25k")
LOCAL_ROOT = Path("/content/deepvoice_dynamic25k")
VOICE_ROOT = LOCAL_ROOT / "voice_kaggle"
FMA_ROOT = LOCAL_ROOT / "fma"
SONICS_ROOT = LOCAL_ROOT / "sonics"
REPO_ROOT = LOCAL_ROOT / "repos"
RUN_ROOT = DRIVE_ROOT / "runs"
MANIFEST_ROOT = DRIVE_ROOT / "manifests"
for directory in (DRIVE_ROOT, LOCAL_ROOT, VOICE_ROOT, FMA_ROOT, SONICS_ROOT, REPO_ROOT, RUN_ROOT, MANIFEST_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

random.seed(CFG.seed)
np.random.seed(CFG.seed)
torch.manual_seed(CFG.seed)
torch.cuda.manual_seed_all(CFG.seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), torch.__version__)
print("source references:", f"{4 * CFG.source_per_pool:,}")
print("dynamic recipes:", f"{sum(CFG.recipe_counts.values()):,}")
print("free disk GB:", round(shutil.disk_usage("/content").free / 1024**3, 1))


## 1. Real/Fake Voice와 FMA 다운로드

Kaggle 데이터셋은 `real/` 9,066개, `fake/` 6,722개 구조이며 약 4.7GB입니다. FMA medium은 small+medium 30초 MP3 25,000개(약 22GiB)와 metadata를 공식 배포 URL에서 받습니다. FMA small만으로는 NoDerivatives를 올바르게 제외한 뒤 6,250곡이 남지 않으므로 medium 묶음을 사용합니다. 압축 파일 hash를 확인하고 재실행 시 기존 파일을 재사용합니다.


In [ ]:
KAGGLE_DATASET = "jayjoshi37/deepfake-audio-dataset-fake-vs-real-speech"
DOWNLOAD_VOICE = True
DOWNLOAD_FMA = True


def configure_kaggle_auth():
    from google.colab import files, userdata
    token = username = key = None
    try:
        token = userdata.get("KAGGLE_API_KEY")
    except Exception:
        pass
    if not token:
        try:
            username = userdata.get("KAGGLE_USERNAME")
            key = userdata.get("KAGGLE_KEY")
        except Exception:
            pass
    if token:
        os.environ["KAGGLE_API_TOKEN"] = token
    elif username and key:
        os.environ["KAGGLE_USERNAME"] = username
        os.environ["KAGGLE_KEY"] = key
    else:
        print("Kaggle Settings에서 받은 kaggle.json을 업로드하세요.")
        uploaded = files.upload()
        if "kaggle.json" not in uploaded:
            raise FileNotFoundError("kaggle.json이 업로드되지 않았습니다.")
        credential_dir = Path("/root/.kaggle")
        credential_dir.mkdir(parents=True, exist_ok=True)
        credential_path = credential_dir / "kaggle.json"
        credential_path.write_bytes(uploaded["kaggle.json"])
        credential_path.chmod(0o600)


def sha1_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha1()
    with Path(path).open("rb") as stream:
        while True:
            chunk = stream.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


voice_audio = [p for p in VOICE_ROOT.rglob("*") if p.is_file() and p.suffix.lower() in AUDIO_SUFFIXES]
if DOWNLOAD_VOICE and not voice_audio:
    configure_kaggle_auth()
    subprocess.run(
        ["kaggle", "datasets", "download", "-d", KAGGLE_DATASET, "-p", str(VOICE_ROOT), "--unzip", "--quiet"],
        check=True,
    )

FMA_FILES = {
    "fma_medium.zip": (
        "https://os.unil.cloud.switch.ch/fma/fma_medium.zip",
        "c67b69ea232021025fca9231fc1c7c1a063ab50b",
    ),
    "fma_metadata.zip": (
        "https://os.unil.cloud.switch.ch/fma/fma_metadata.zip",
        "f0df49ffe5f2a6008d7dc83c6915b31835dfe733",
    ),
}
if DOWNLOAD_FMA:
    for filename, (url, expected_sha1) in FMA_FILES.items():
        archive_path = FMA_ROOT / filename
        if not archive_path.exists():
            subprocess.run(["wget", "-q", "--show-progress", "-O", str(archive_path), url], check=True)
        actual_sha1 = sha1_file(archive_path)
        if actual_sha1 != expected_sha1:
            raise RuntimeError(f"FMA hash mismatch: {filename} {actual_sha1}")
        marker = FMA_ROOT / (filename + ".extracted")
        if not marker.exists():
            subprocess.run(["unzip", "-q", "-o", str(archive_path), "-d", str(FMA_ROOT)], check=True)
            marker.write_text(actual_sha1, encoding="utf-8")

voice_audio = [p for p in VOICE_ROOT.rglob("*") if p.is_file() and p.suffix.lower() in AUDIO_SUFFIXES]
fma_audio = [p for p in (FMA_ROOT / "fma_medium").rglob("*.mp3") if p.is_file()]
print("voice audio:", f"{len(voice_audio):,}")
print("FMA medium audio:", f"{len(fma_audio):,}")
if len(voice_audio) < 2 * CFG.source_per_pool:
    raise RuntimeError("Kaggle real/fake 음성 파일이 충분하지 않습니다.")
if len(fma_audio) < CFG.source_per_pool:
    raise RuntimeError("FMA real music 파일이 충분하지 않습니다.")


## 2. SONICS fake music 다운로드

[SONICS 공식 데이터셋](https://huggingface.co/datasets/awsaf49/sonics)은 49,074개의 Suno/Udio 가짜 곡을 10개 ZIP으로 제공합니다. 각 ZIP에는 5,000곡이 들어 있으므로 `part_01.zip`과 `part_02.zip`만 내려받아 10,000곡 중 6,250곡을 선택합니다. 전체 32.6GB 저장소를 받을 필요가 없습니다.

SONICS는 가짜 곡 오디오만 제공하므로 real music은 FMA를 계속 사용합니다. 데이터셋 라이선스는 CC BY-NC 4.0이며, metadata의 `source`, `algorithm`, `label`, `split`, `no_vocal`을 source manifest에 보존합니다.


In [ ]:
from huggingface_hub import hf_hub_download

SONICS_REPO = "awsaf49/sonics"
SONICS_REVISION = "3788dca9f9f11ad92e9097ef4b58eee247661e7f"
SONICS_PARTS = ["fake_songs/part_01.zip", "fake_songs/part_02.zip"]
DOWNLOAD_SONICS = True
DELETE_SONICS_ARCHIVES_AFTER_EXTRACT = True
SONICS_METADATA_PATH = SONICS_ROOT / "fake_songs.csv"
SONICS_AUDIO_ROOT = SONICS_ROOT / "fake_songs"
SONICS_ROOT.mkdir(parents=True, exist_ok=True)


def download_sonics_file(filename):
    return Path(hf_hub_download(
        repo_id=SONICS_REPO,
        repo_type="dataset",
        filename=filename,
        revision=SONICS_REVISION,
        local_dir=str(SONICS_ROOT),
    ))


if DOWNLOAD_SONICS:
    if not SONICS_METADATA_PATH.exists():
        downloaded_metadata = download_sonics_file("fake_songs.csv")
        if downloaded_metadata.resolve() != SONICS_METADATA_PATH.resolve():
            shutil.copy2(downloaded_metadata, SONICS_METADATA_PATH)

    for part_name in SONICS_PARTS:
        part_stem = Path(part_name).stem
        marker = SONICS_ROOT / f".{part_stem}.extracted"
        if marker.exists():
            print("reuse extracted:", part_name)
            continue
        archive_path = download_sonics_file(part_name)
        with zipfile.ZipFile(archive_path) as archive:
            members = [info for info in archive.infolist() if not info.is_dir()]
            if len(members) != 5_000:
                raise RuntimeError(f"SONICS {part_name}: expected 5,000 files, got {len(members):,}")
            archive.extractall(SONICS_ROOT)
        marker.write_text(f"{SONICS_REVISION}\n{len(members)} files\n", encoding="utf-8")
        if DELETE_SONICS_ARCHIVES_AFTER_EXTRACT and archive_path.exists():
            archive_path.unlink()
elif not SONICS_METADATA_PATH.exists() or not SONICS_AUDIO_ROOT.exists():
    raise FileNotFoundError("DOWNLOAD_SONICS=False이면 SONICS metadata와 fake_songs 폴더를 직접 준비해야 합니다.")

sonics_audio = sorted(
    path for path in SONICS_AUDIO_ROOT.rglob("*")
    if path.is_file() and path.suffix.lower() in AUDIO_SUFFIXES and path.stat().st_size > 0
)
sonics_metadata = pd.read_csv(
    SONICS_METADATA_PATH,
    usecols=["filename", "algorithm", "source", "label", "target", "skip_time", "no_vocal", "split"],
    low_memory=False,
)
sonics_metadata = sonics_metadata[sonics_metadata["target"].eq(1)].copy()
sonics_metadata["file_stem"] = sonics_metadata["filename"].astype(str).map(lambda value: Path(value).stem)
if sonics_metadata["no_vocal"].dtype != bool:
    sonics_metadata["no_vocal"] = (
        sonics_metadata["no_vocal"].astype(str).str.strip().str.lower().isin({"true", "1", "yes"})
    )
sonics_metadata = sonics_metadata.drop_duplicates("file_stem", keep="last")
sonics_metadata_lookup = sonics_metadata.set_index("file_stem").to_dict("index")
sonics_candidates = [
    str(path.resolve()) for path in sonics_audio if path.stem in sonics_metadata_lookup
]
print({
    "SONICS extracted audio": len(sonics_audio),
    "metadata fake rows": len(sonics_metadata),
    "matched candidates": len(sonics_candidates),
    "sources": sonics_metadata.loc[
        sonics_metadata["file_stem"].isin({Path(path).stem for path in sonics_candidates}), "source"
    ].value_counts().to_dict(),
})
if len(sonics_candidates) < CFG.source_per_pool:
    raise RuntimeError(f"SONICS fake song이 {CFG.source_per_pool:,}개 필요합니다.")


## 3. 네 소스 풀에서 각각 정확히 6,250개 선택

Kaggle 폴더의 `real/`, `fake/`를 음성 라벨로 사용합니다. FMA는 metadata에서 small+medium subset을 확인하고 `NoDerivatives` 계열을 제외합니다. SONICS는 다운로드한 10,000곡 중 metadata와 일치하는 가짜 곡을 선택합니다. 모든 선택은 stable hash 정렬을 사용해 재실행해도 동일합니다.


In [ ]:
def stable_int(text):
    return int(hashlib.sha256(str(text).encode("utf-8")).hexdigest()[:16], 16)


def select_exact(paths, count, namespace):
    unique_paths = sorted({str(Path(path).resolve()) for path in paths})
    ordered = sorted(unique_paths, key=lambda value: stable_int(f"{CFG.seed}|{namespace}|{value}"))
    if len(ordered) < count:
        raise RuntimeError(f"{namespace}: {count:,}개 필요, {len(ordered):,}개 발견")
    return ordered[:count]


real_voice_all, fake_voice_all, unresolved_voice = [], [], []
for path in voice_audio:
    parts = {part.lower() for part in path.parts}
    if "real" in parts and "fake" not in parts:
        real_voice_all.append(path)
    elif "fake" in parts and "real" not in parts:
        fake_voice_all.append(path)
    else:
        unresolved_voice.append(path)
print({"real_voice": len(real_voice_all), "fake_voice": len(fake_voice_all), "unresolved": len(unresolved_voice)})

tracks_path = FMA_ROOT / "fma_metadata" / "tracks.csv"
tracks = pd.read_csv(tracks_path, index_col=0, header=[0, 1], low_memory=False)
small_medium_tracks = tracks[
    tracks[("set", "subset")].astype(str).isin({"small", "medium"})
].copy()

def fma_track_path(track_id):
    track_id = int(track_id)
    return FMA_ROOT / "fma_medium" / f"{track_id:06d}"[:3] / f"{track_id:06d}.mp3"

def fma_license_allowed(value):
    value = str(value).lower()
    if not value or value == "nan":
        return False
    compact = re.sub(r"[^a-z0-9]+", "", value)
    normalized = re.sub(r"[^a-z0-9]+", " ", value)
    no_derivatives = (
        "noderivative" in compact
        or "noderivs" in compact
        or "musicsharing" in compact
        or re.search(r"\bby\s+(?:nc\s+)?nd\b", normalized) is not None
    )
    return not no_derivatives

fma_candidates = []
fma_license_lookup = {}
for track_id, row in small_medium_tracks.iterrows():
    path = fma_track_path(track_id)
    license_value = row.get(("track", "license"), "")
    if path.is_file() and path.stat().st_size > 0 and fma_license_allowed(license_value):
        fma_candidates.append(path)
        fma_license_lookup[str(path.resolve())] = str(license_value)

selected = {
    "real_voice": select_exact(real_voice_all, CFG.source_per_pool, "real_voice"),
    "fake_voice": select_exact(fake_voice_all, CFG.source_per_pool, "fake_voice"),
    "real_music": select_exact(fma_candidates, CFG.source_per_pool, "real_music"),
    "fake_music": select_exact(sonics_candidates, CFG.source_per_pool, "fake_music"),
}
assert sum(map(len, selected.values())) == 25_000
if set(selected["real_voice"]) & set(selected["fake_voice"]):
    raise RuntimeError("real/fake voice pool overlap")
display(pd.DataFrame({name: [len(paths)] for name, paths in selected.items()}))
print("FMA eligible after license filter:", f"{len(fma_candidates):,}")


## 4. FMA 보컬 스크리닝

PANNs Cnn14의 AudioSet `Speech`, `Singing`, `Choir`, `Vocal music` 계열 점수로 FMA track의 보컬 가능성을 기록합니다. 모델이 music-only로 잘못 학습하지 않도록, 검출된 track은 real music과 real voice가 함께 존재하는 source로 취급합니다.

스크리닝 결과는 Drive CSV에 누적 저장되어 중단 후 이어집니다. `RUN_FMA_VOCAL_SCREEN=False`는 빠른 디버그에만 사용하세요.


In [ ]:
RUN_FMA_VOCAL_SCREEN = True
FMA_VOCAL_THRESHOLD = 0.20
FMA_SCREEN_PATH = MANIFEST_ROOT / "fma_panns_vocal_screen.csv"

if FMA_SCREEN_PATH.exists():
    fma_screen = pd.read_csv(FMA_SCREEN_PATH)
else:
    fma_screen = pd.DataFrame(columns=["path", "vocal_score", "music_score", "contains_voice", "screen_ok"])

def as_boolean(series):
    if series.dtype == bool:
        return series
    return series.astype(str).str.strip().str.lower().isin({"true", "1", "yes"})

if RUN_FMA_VOCAL_SCREEN:
    from panns_inference import AudioTagging, labels as panns_labels

    vocal_names = [
        "Speech", "Male speech, man speaking", "Female speech, woman speaking",
        "Conversation", "Narration, monologue", "Singing", "Choir", "Vocal music",
    ]
    vocal_indices = [panns_labels.index(name) for name in vocal_names if name in panns_labels]
    music_index = panns_labels.index("Music")
    completed = set(fma_screen.loc[as_boolean(fma_screen["screen_ok"]), "path"].astype(str)) if len(fma_screen) else set()
    missing = [path for path in selected["real_music"] if path not in completed]

    def load_for_panns(path):
        audio, _ = librosa.load(path, sr=32_000, mono=True, duration=CFG.panns_seconds)
        target = 32_000 * CFG.panns_seconds
        if len(audio) < target:
            audio = np.pad(audio, (0, target - len(audio)))
        return np.asarray(audio[:target], dtype=np.float32)

    new_rows = []
    if missing:
        checkpoint_path = DRIVE_ROOT / "panns" / "Cnn14_mAP=0.431.pth"
        checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
        tagger = AudioTagging(checkpoint_path=str(checkpoint_path), device=str(DEVICE))
    for offset in tqdm(range(0, len(missing), CFG.panns_batch), desc="FMA PANNs vocal screen"):
        batch_paths = missing[offset:offset + CFG.panns_batch]
        batch_audio, ok_flags = [], []
        for path in batch_paths:
            try:
                batch_audio.append(load_for_panns(path))
                ok_flags.append(True)
            except Exception:
                batch_audio.append(np.zeros(32_000 * CFG.panns_seconds, dtype=np.float32))
                ok_flags.append(False)
        clipwise, _ = tagger.inference(np.stack(batch_audio))
        for path, scores, ok in zip(batch_paths, clipwise, ok_flags):
            vocal_score = float(np.max(scores[vocal_indices])) if vocal_indices else 0.0
            new_rows.append({
                "path": path, "vocal_score": vocal_score,
                "music_score": float(scores[music_index]),
                "contains_voice": bool(vocal_score >= FMA_VOCAL_THRESHOLD),
                "screen_ok": bool(ok),
            })
        if len(new_rows) >= 128 or offset + CFG.panns_batch >= len(missing):
            fma_screen = pd.concat([fma_screen, pd.DataFrame(new_rows)], ignore_index=True)
            fma_screen = fma_screen.drop_duplicates("path", keep="last")
            fma_screen.to_csv(FMA_SCREEN_PATH, index=False, encoding="utf-8")
            new_rows = []
    if missing:
        del tagger
    gc.collect()
    torch.cuda.empty_cache()
else:
    fma_screen = pd.DataFrame({
        "path": selected["real_music"], "vocal_score": 0.0, "music_score": np.nan,
        "contains_voice": False, "screen_ok": False,
    })
    print("경고: FMA 보컬 스크리닝을 끈 상태입니다. music-only 라벨 노이즈가 생길 수 있습니다.")

fma_screen = fma_screen[fma_screen["path"].isin(selected["real_music"])].copy()
fma_screen["contains_voice"] = as_boolean(fma_screen["contains_voice"])
fma_screen["screen_ok"] = as_boolean(fma_screen["screen_ok"])
print("FMA screened:", len(fma_screen), "vocal-like:", int(fma_screen["contains_voice"].astype(bool).sum()))
display(fma_screen.describe(include="all").T)


## 5. Source manifest와 누수 없는 split

각 풀의 6,250개를 5,000/625/625로 정확히 나눕니다. 동일 source path가 split을 넘지 않는지 검사하고, 데이터 출처·license·FMA 보컬 스크리닝 값을 manifest에 보존합니다.


In [ ]:
fma_voice_lookup = dict(zip(fma_screen["path"].astype(str), fma_screen["contains_voice"].astype(bool)))
rows = []
for pool, paths in selected.items():
    ordered = sorted(paths, key=lambda value: stable_int(f"split|{CFG.seed}|{pool}|{value}"))
    boundaries = {}
    start = 0
    for split_name, count in CFG.source_split_counts.items():
        boundaries[split_name] = ordered[start:start + count]
        start += count
    for split_name, split_paths in boundaries.items():
        for path in split_paths:
            sonics_record = sonics_metadata_lookup.get(Path(path).stem, {}) if pool == "fake_music" else {}
            if pool == "real_music":
                contains_voice = bool(fma_voice_lookup.get(path, False))
            elif pool == "fake_music":
                contains_voice = not bool(sonics_record.get("no_vocal", False))
            else:
                contains_voice = pool.endswith("voice")
            rows.append({
                "source_id": hashlib.sha256(f"{pool}|{path}".encode()).hexdigest()[:20],
                "pool": pool, "split": split_name, "path": path,
                "contains_voice": contains_voice,
                "voice_fake": int(pool == "fake_voice" or (pool == "fake_music" and contains_voice)),
                "contains_music": pool.endswith("music"),
                "music_fake": int(pool == "fake_music"),
                "license": (
                    "CC-BY-SA-4.0" if pool.endswith("voice") else
                    fma_license_lookup.get(path, "FMA artist-selected") if pool == "real_music" else
                    "CC-BY-NC-4.0"
                ),
                "origin": (
                    KAGGLE_DATASET if pool.endswith("voice") else
                    "FMA medium" if pool == "real_music" else SONICS_REPO
                ),
                "source_detail": str(sonics_record.get("source", "")),
                "generation_algorithm": str(sonics_record.get("algorithm", "")),
                "sonics_label": str(sonics_record.get("label", "")),
                "sonics_original_split": str(sonics_record.get("split", "")),
            })
source_manifest = pd.DataFrame(rows)
assert len(source_manifest) == 25_000
assert source_manifest["source_id"].is_unique
assert source_manifest.groupby("pool").size().eq(CFG.source_per_pool).all()
assert source_manifest.groupby(["pool", "split"]).size().to_dict() == {
    (pool, split_name): count
    for pool in selected for split_name, count in CFG.source_split_counts.items()
}
if source_manifest.groupby("path")["split"].nunique().max() != 1:
    raise RuntimeError("source path가 여러 split에 존재합니다.")
SOURCE_MANIFEST_PATH = MANIFEST_ROOT / "source_manifest_25000.csv"
source_manifest.to_csv(SOURCE_MANIFEST_PATH, index=False, encoding="utf-8")
display(pd.crosstab(source_manifest["pool"], source_manifest["split"], margins=True))
display(source_manifest.groupby(["pool", "license"]).size().rename("count").reset_index())
display(source_manifest[source_manifest["pool"].eq("fake_music")][
    ["source_detail", "generation_algorithm", "sonics_label", "contains_voice"]
].value_counts().rename("count").reset_index().head(30))
print("saved:", SOURCE_MANIFEST_PATH)


## 6. 정확히 25,000개 Dynamic Mix recipe

여덟 조합을 균형 배치합니다: RV, FV, RM, FM, RV+RM, FV+RM, RV+FM, FV+FM. Recipe CSV에는 실제 waveform 대신 split, 조합, seed를 저장합니다. Train은 `epoch`을 seed에 포함해 매 epoch 다른 source/crop/SNR/layout을 만들고 validation/audit은 고정합니다.


In [ ]:
RECIPE_TYPES = ["rv", "fv", "rm", "fm", "rv_rm", "fv_rm", "rv_fm", "fv_fm"]


def build_recipe_manifest():
    recipe_rows = []
    for split_name, count in CFG.recipe_counts.items():
        recipe_types = [RECIPE_TYPES[index % len(RECIPE_TYPES)] for index in range(count)]
        random.Random(CFG.seed + stable_int(split_name)).shuffle(recipe_types)
        for index, recipe_type in enumerate(recipe_types):
            recipe_rows.append({
                "recipe_id": f"{split_name}_{index:05d}",
                "split": split_name, "recipe_type": recipe_type,
                "seed": stable_int(f"recipe|{CFG.seed}|{split_name}|{index}|{recipe_type}") % (2**31 - 1),
            })
    return pd.DataFrame(recipe_rows)


recipe_manifest = build_recipe_manifest()
assert len(recipe_manifest) == 25_000
assert recipe_manifest["recipe_id"].is_unique
assert recipe_manifest.groupby("split").size().to_dict() == CFG.recipe_counts
RECIPE_MANIFEST_PATH = MANIFEST_ROOT / "dynamic_mix_recipes_25000.csv"
recipe_manifest.to_csv(RECIPE_MANIFEST_PATH, index=False, encoding="utf-8")
display(pd.crosstab(recipe_manifest["recipe_type"], recipe_manifest["split"], margins=True))
print("saved:", RECIPE_MANIFEST_PATH)


## 7. 공식 AASIST·RawBoost 소스와 오디오 로더

SoundFile → librosa → FFmpeg 순서로 WAV/MP3/FLAC 등을 처리합니다. 긴 파일은 random crop, 짧은 파일은 반복 후 crop합니다. RawBoost는 train 최종 mixture에만 적용합니다.


In [ ]:
repositories = {
    "aasist": "https://github.com/clovaai/aasist.git",
    "rawboost": "https://github.com/TakHemlata/RawBoost-antispoofing.git",
}
repo_commits = {}
for name, url in repositories.items():
    destination = REPO_ROOT / name
    if not destination.exists():
        subprocess.run(["git", "clone", "--depth", "1", url, str(destination)], check=True)
    repo_commits[name] = subprocess.check_output(
        ["git", "-C", str(destination), "rev-parse", "HEAD"], text=True,
    ).strip()
print(repo_commits)

rawboost_path = REPO_ROOT / "rawboost" / "RawBoost.py"
rawboost_spec = importlib.util.spec_from_file_location("official_rawboost", rawboost_path)
RAWBOOST = importlib.util.module_from_spec(rawboost_spec)
assert rawboost_spec.loader is not None
rawboost_spec.loader.exec_module(RAWBOOST)


def decode_audio(path):
    path = str(path)
    try:
        audio, sample_rate = sf.read(path, dtype="float32", always_2d=True)
        waveform = torch.from_numpy(audio.mean(axis=1))
    except Exception:
        try:
            audio, sample_rate = librosa.load(path, sr=None, mono=True)
            waveform = torch.from_numpy(np.asarray(audio, dtype=np.float32))
        except Exception:
            decoded = subprocess.run(
                [
                    "ffmpeg", "-hide_banner", "-loglevel", "error", "-i", path,
                    "-ac", "1", "-ar", str(CFG.sample_rate), "-f", "f32le", "pipe:1",
                ],
                stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True, timeout=90,
            )
            waveform = torch.from_numpy(np.frombuffer(decoded.stdout, dtype="<f4").copy())
            sample_rate = CFG.sample_rate
    if waveform.numel() == 0:
        raise ValueError(f"empty audio: {path}")
    if sample_rate != CFG.sample_rate:
        waveform = torchaudio.functional.resample(waveform, sample_rate, CFG.sample_rate)
    waveform = waveform.float().nan_to_num().clamp(-1, 1)
    if waveform.numel() == 0:
        raise ValueError(f"empty after resample: {path}")
    return waveform


def crop_or_repeat(waveform, rng, training):
    if waveform.numel() < CFG.clip_samples:
        waveform = waveform.repeat(math.ceil(CFG.clip_samples / waveform.numel()))
    maximum_start = waveform.numel() - CFG.clip_samples
    start = rng.randint(0, maximum_start) if training and maximum_start else maximum_start // 2
    return waveform[start:start + CFG.clip_samples].clone()


## 8. DynamicMixDataset

두 성분의 RMS를 맞춘 뒤 voice-to-music SNR을 -12~+12dB에서 샘플링합니다. overlap/partial/sequential layout을 섞고, 최종 peak도 0.65~0.98 사이로 바꿔 “크면 mixed”라는 지름길을 막습니다.


In [ ]:
RECIPE_COMPONENTS = {
    "rv": ["real_voice"], "fv": ["fake_voice"],
    "rm": ["real_music"], "fm": ["fake_music"],
    "rv_rm": ["real_voice", "real_music"],
    "fv_rm": ["fake_voice", "real_music"],
    "rv_fm": ["real_voice", "fake_music"],
    "fv_fm": ["fake_voice", "fake_music"],
}


def rms_normalize(waveform, target_db):
    rms = waveform.square().mean().clamp_min(1e-8).sqrt()
    target = 10 ** (target_db / 20)
    return waveform * (target / rms)


def apply_temporal_layout(waveforms, rng):
    if len(waveforms) < 2:
        return waveforms, "single"
    choice = rng.random()
    if choice < 0.65:
        return waveforms, "overlap"
    length = CFG.clip_samples
    if choice < 0.85:
        result = []
        for waveform in waveforms:
            active = rng.randint(CFG.sample_rate, length)
            start = rng.randint(0, length - active)
            mask = torch.zeros(length)
            mask[start:start + active] = 1
            result.append(waveform * mask)
        return result, "partial"
    boundary = rng.randint(int(0.35 * length), int(0.65 * length))
    first_mask = torch.zeros(length); first_mask[:boundary] = 1
    second_mask = 1 - first_mask
    return [waveforms[0] * first_mask, waveforms[1] * second_mask], "sequential"


def communication_augment(waveform, rng):
    mode = rng.choice(["telephone", "mulaw", "noise", "gain_clip"])
    if mode == "telephone":
        waveform = torchaudio.functional.resample(waveform, CFG.sample_rate, 8000)
        waveform = torchaudio.functional.resample(waveform, 8000, CFG.sample_rate)
    elif mode == "mulaw":
        encoded = torchaudio.functional.mu_law_encoding(waveform.clamp(-1, 1), 256)
        waveform = torchaudio.functional.mu_law_decoding(encoded, 256)
    elif mode == "noise":
        power = waveform.square().mean().clamp_min(1e-8)
        snr_db = rng.uniform(12, 35)
        generator = torch.Generator().manual_seed(rng.randrange(2**31 - 1))
        noise = torch.randn(waveform.shape, generator=generator, dtype=waveform.dtype)
        waveform = waveform + noise * (power / 10 ** (snr_db / 10)).sqrt()
    else:
        waveform = waveform * 10 ** (rng.uniform(-8, 5) / 20)
        limit = rng.uniform(0.35, 0.95)
        waveform = waveform.clamp(-limit, limit) / limit
    return waveform


class DynamicMixDataset(Dataset):
    def __init__(self, recipes, sources, training=False, rawboost_p=0.0, communication_p=0.0):
        self.recipes = recipes.reset_index(drop=True)
        self.training = training
        self.rawboost_p = rawboost_p
        self.communication_p = communication_p
        self.epoch = 0
        self.pools = {
            pool: frame.to_dict("records")
            for pool, frame in sources.groupby("pool", sort=False)
        }
        for pool in RECIPE_COMPONENTS.values():
            for name in pool:
                if not self.pools.get(name):
                    raise RuntimeError(f"empty source pool: {name}")

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __len__(self):
        return len(self.recipes)

    def _load_from_pool(self, pool_name, rng):
        errors = []
        for _ in range(5):
            entry = self.pools[pool_name][rng.randrange(len(self.pools[pool_name]))]
            try:
                waveform = crop_or_repeat(decode_audio(entry["path"]), rng, self.training)
                return waveform, entry
            except Exception as exc:
                errors.append(f"{entry['path']}: {repr(exc)}")
        raise RuntimeError(" | ".join(errors))

    def __getitem__(self, index):
        recipe = self.recipes.iloc[index]
        epoch = self.epoch if self.training else 0
        rng = random.Random(stable_int(f"mix|{recipe.seed}|{epoch}|{index}"))
        components = RECIPE_COMPONENTS[recipe.recipe_type]
        waveforms, entries = [], []
        try:
            for pool_name in components:
                waveform, entry = self._load_from_pool(pool_name, rng)
                waveforms.append(waveform)
                entries.append((pool_name, entry))
        except Exception as exc:
            return {
                "audio": torch.zeros(CFG.clip_samples), "target": torch.zeros(5),
                "mask": torch.ones(5), "id": recipe.recipe_id,
                "recipe_type": recipe.recipe_type, "valid": torch.tensor(False),
                "error": repr(exc), "layout": "error",
            }

        if len(waveforms) == 2:
            snr_db = rng.uniform(-12, 12)
            waveforms[0] = rms_normalize(waveforms[0], -22 + snr_db / 2)
            waveforms[1] = rms_normalize(waveforms[1], -22 - snr_db / 2)
        else:
            waveforms[0] = rms_normalize(waveforms[0], rng.uniform(-27, -17))
        waveforms, layout = apply_temporal_layout(waveforms, rng)
        mixed = torch.stack(waveforms).sum(0)

        target = torch.zeros(5, dtype=torch.float32)
        voice_present = voice_fake = music_present = music_fake = 0
        for pool_name, entry in entries:
            if pool_name.endswith("voice"):
                voice_present = 1
                voice_fake = max(voice_fake, int(pool_name == "fake_voice"))
            if pool_name.endswith("music"):
                music_present = 1
                music_fake = max(music_fake, int(pool_name == "fake_music"))
                if bool(entry.get("contains_voice", False)):
                    voice_present = 1
                    voice_fake = max(voice_fake, int(pool_name == "fake_music"))
        file_fake = max(voice_fake, music_fake)
        target[:] = torch.tensor([file_fake, voice_fake, music_fake, voice_present, music_present])
        mask = torch.tensor([1, voice_present, music_present, 1, 1], dtype=torch.float32)

        if self.training and rng.random() < self.rawboost_p:
            values = mixed.numpy()
            values = RAWBOOST.LnL_convolutive_noise(
                values, N_f=5, nBands=5, minF=20, maxF=8000,
                minBW=100, maxBW=1000, minCoeff=10, maxCoeff=100,
                minG=0, maxG=0, minBiasLinNonLin=5, maxBiasLinNonLin=20,
                fs=CFG.sample_rate,
            )
            values = RAWBOOST.ISD_additive_noise(values, P=10, g_sd=2)
            mixed = torch.from_numpy(np.asarray(RAWBOOST.normWav(values, 0), dtype=np.float32))
        if self.training and rng.random() < self.communication_p:
            mixed = communication_augment(mixed, rng)

        peak = mixed.abs().max().clamp_min(1e-8)
        mixed = mixed / peak * rng.uniform(0.65, 0.98)
        if not torch.isfinite(mixed).all():
            return {
                "audio": torch.zeros(CFG.clip_samples), "target": target, "mask": mask,
                "id": recipe.recipe_id, "recipe_type": recipe.recipe_type,
                "valid": torch.tensor(False), "error": "NaN/Inf after mixing", "layout": layout,
            }
        return {
            "audio": mixed.float().clamp(-1, 1), "target": target, "mask": mask,
            "id": recipe.recipe_id, "recipe_type": recipe.recipe_type,
            "valid": torch.tensor(True), "error": "", "layout": layout,
        }


source_by_split = {
    split_name: source_manifest[source_manifest["split"] == split_name].copy()
    for split_name in CFG.source_split_counts
}
recipe_by_split = {
    split_name: recipe_manifest[recipe_manifest["split"] == split_name].copy()
    for split_name in CFG.recipe_counts
}
preview_dataset = DynamicMixDataset(recipe_by_split["validation"].head(8), source_by_split["validation"])
preview_rows = []
for index in range(len(preview_dataset)):
    item = preview_dataset[index]
    preview_rows.append({"id": item["id"], "type": item["recipe_type"], **dict(zip(DACON_TRUTH_COLUMNS, item["target"].tolist()))})
display(pd.DataFrame(preview_rows))


## 9. DACON 공식 지표와 masked multi-task loss

[DACON 공식 평가 페이지](https://dacon.io/competitions/official/236749/overview/evaluation)의 계산을 그대로 사용합니다.

- `ADS = 0.5 × (1 - File EER) + 0.2 × (1 - Voice EER) + 0.3 × (1 - Music EER)`
- `CPS = 0.5 × Voice Presence ROC-AUC + 0.5 × Music Presence ROC-AUC`
- `Score = 0.9 × ADS + 0.1 × CPS` (높을수록 좋음)
- FAKE가 양성 클래스 `1`이며, Voice/Music EER은 해당 성분이 존재하는 샘플에서만 계산합니다.

Loss weight는 최종 Score의 각 항 가중치를 그대로 펼친 값입니다: File 0.45, Voice Fake 0.18, Music Fake 0.27, Voice Presence 0.05, Music Presence 0.05. 성분이 없는 fake head는 loss mask에서 제외합니다.


In [ ]:
OFFICIAL_METRIC_WEIGHTS = {
    "score_ads": 0.9,
    "score_cps": 0.1,
    "ads_file": 0.5,
    "ads_voice": 0.2,
    "ads_music": 0.3,
    "cps_voice_presence": 0.5,
    "cps_music_presence": 0.5,
}


def _official_binary_inputs(y_true, y_score, metric_name):
    y_true = np.asarray(y_true, dtype=int)
    y_score = np.asarray(y_score, dtype=float)
    if y_true.shape != y_score.shape or y_true.size == 0:
        raise ValueError(f"{metric_name}: shape/empty input error")
    if not np.isfinite(y_true).all() or not np.isfinite(y_score).all():
        raise ValueError(f"{metric_name}: NaN/Inf input")
    if np.unique(y_true).size < 2:
        raise ValueError(f"{metric_name}: positive/negative classes are both required")
    return y_true, y_score


def equal_error_rate(y_true, y_score):
    y_true, y_score = _official_binary_inputs(y_true, y_score, "EER")
    # DACON 평가 페이지에 공개된 EER 구현과 동일합니다.
    fpr, tpr, _ = roc_curve(y_true, y_score, pos_label=1, drop_intermediate=False)
    fnr = 1 - tpr
    idx = np.argmin(np.abs(fpr - fnr))
    eer = (fpr[idx] + fnr[idx]) / 2
    return float(eer)


def official_roc_auc(y_true, y_score):
    y_true, y_score = _official_binary_inputs(y_true, y_score, "ROC-AUC")
    return float(roc_auc_score(y_true, y_score))


def dacon_official_score(y_true, y_pred):
    file_eer = equal_error_rate(y_true["FILE_FAKE"], y_pred["FILE_FAKE_PROB"])
    voice_mask = y_true["VOICE_PRESENT"].eq(1)
    music_mask = y_true["MUSIC_PRESENT"].eq(1)
    voice_eer = equal_error_rate(y_true.loc[voice_mask, "VOICE_FAKE"], y_pred.loc[voice_mask, "VOICE_FAKE_PROB"])
    music_eer = equal_error_rate(y_true.loc[music_mask, "MUSIC_FAKE"], y_pred.loc[music_mask, "MUSIC_FAKE_PROB"])
    voice_auc = official_roc_auc(y_true["VOICE_PRESENT"], y_pred["VOICE_PRESENT_PROB"])
    music_auc = official_roc_auc(y_true["MUSIC_PRESENT"], y_pred["MUSIC_PRESENT_PROB"])
    ads = (
        OFFICIAL_METRIC_WEIGHTS["ads_file"] * (1 - file_eer)
        + OFFICIAL_METRIC_WEIGHTS["ads_voice"] * (1 - voice_eer)
        + OFFICIAL_METRIC_WEIGHTS["ads_music"] * (1 - music_eer)
    )
    cps = (
        OFFICIAL_METRIC_WEIGHTS["cps_voice_presence"] * voice_auc
        + OFFICIAL_METRIC_WEIGHTS["cps_music_presence"] * music_auc
    )
    score = (
        OFFICIAL_METRIC_WEIGHTS["score_ads"] * ads
        + OFFICIAL_METRIC_WEIGHTS["score_cps"] * cps
    )
    return {
        "file_eer": file_eer, "voice_eer": voice_eer, "music_eer": music_eer,
        "voice_presence_auc": voice_auc, "music_presence_auc": music_auc,
        "ads": ads, "cps": cps, "score": score,
    }


def masked_multitask_loss(logits, targets, masks):
    loss = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
    weights = HEAD_WEIGHTS.to(logits.device).unsqueeze(0) * masks
    return (loss * weights).sum() / weights.sum().clamp_min(1e-8)


def report_from_arrays(targets, predictions):
    y_true = pd.DataFrame(targets, columns=DACON_TRUTH_COLUMNS)
    y_pred = pd.DataFrame(predictions, columns=DACON_PROBABILITY_COLUMNS)
    return dacon_official_score(y_true, y_pred)


## 10. 모델: Tiny Log-Mel와 공식 AASIST

모든 모델은 logits 5개를 출력합니다. 공식 AASIST의 마지막 `out_layer`만 5개로 교체하고 나머지 구조는 그대로 사용합니다.


In [ ]:
class TinyLogMel(nn.Module):
    def __init__(self, dropout=0.25):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=CFG.sample_rate, n_fft=1024, win_length=400,
            hop_length=160, n_mels=96, f_min=20, f_max=7600,
        )
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.SiLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.SiLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.SiLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
        )
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(128, 5))

    def forward(self, audio):
        feature = torch.log(self.mel(audio).clamp_min(1e-6))
        feature = (feature - feature.mean((-2, -1), keepdim=True)) / (feature.std((-2, -1), keepdim=True) + 1e-5)
        return self.head(self.encoder(feature.unsqueeze(1)))


def official_aasist(freq_aug=False):
    repo = REPO_ROOT / "aasist"
    with (repo / "config" / "AASIST.conf").open(encoding="utf-8") as file:
        model_config = json.load(file)["model_config"]
    if str(repo) not in sys.path:
        sys.path.insert(0, str(repo))
    from models.AASIST import Model as AASISTModel

    class Wrapper(nn.Module):
        def __init__(self):
            super().__init__()
            self.net = AASISTModel(model_config)
            if not hasattr(self.net, "out_layer"):
                raise AttributeError("AASIST out_layer를 찾을 수 없습니다.")
            self.net.out_layer = nn.Linear(self.net.out_layer.in_features, 5)

        def forward(self, audio):
            _, logits = self.net(audio, Freq_aug=freq_aug and self.training)
            return logits

    return Wrapper()


## 11. 모델: XLS-R pooling / dual graph / TCM / spectral fusion

기존 기술 스택을 5-head multi-task classifier로 확장합니다. Hugging Face backbone은 Colab 학습 중 다운로드되며, 기본 실행은 제출 패키징이 간단한 AASIST+RawBoost입니다.


In [ ]:
from transformers import AutoModel

XLSR_MODEL = "facebook/wav2vec2-xls-r-300m"


class SSLBase(nn.Module):
    def __init__(self, model_name, freeze=True):
        super().__init__()
        self.ssl = AutoModel.from_pretrained(model_name)
        self.hidden = self.ssl.config.hidden_size
        if freeze:
            for parameter in self.ssl.parameters():
                parameter.requires_grad = False

    def features(self, audio):
        audio = (audio - audio.mean(1, keepdim=True)) / (audio.std(1, keepdim=True) + 1e-5)
        if any(parameter.requires_grad for parameter in self.ssl.parameters()):
            return self.ssl(audio).last_hidden_state
        with torch.no_grad():
            return self.ssl(audio).last_hidden_state

    def unfreeze_last(self, count=4):
        for parameter in self.ssl.parameters():
            parameter.requires_grad = False
        for layer in self.ssl.encoder.layers[-count:]:
            for parameter in layer.parameters():
                parameter.requires_grad = True


class AttentivePool(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.attention = nn.Sequential(nn.Linear(dim, dim // 2), nn.Tanh(), nn.Linear(dim // 2, 1))

    def forward(self, x):
        weights = torch.softmax(self.attention(x), dim=1)
        mean = (x * weights).sum(1)
        std = (weights * (x - mean[:, None]).square()).sum(1).clamp_min(1e-6).sqrt()
        return torch.cat([mean, std], dim=-1)


class XLSRPool(SSLBase):
    def __init__(self, model_name, freeze=True, dropout=0.2):
        super().__init__(model_name, freeze)
        self.pool = AttentivePool(self.hidden)
        self.head = nn.Sequential(nn.LayerNorm(self.hidden * 2), nn.Dropout(dropout), nn.Linear(self.hidden * 2, 256), nn.GELU(), nn.Linear(256, 5))

    def forward(self, audio):
        return self.head(self.pool(self.features(audio)))


class AttentionBlock(nn.Module):
    def __init__(self, dim, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attention = nn.MultiheadAttention(dim, 4, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.ff = nn.Sequential(nn.Linear(dim, dim * 4), nn.GELU(), nn.Dropout(dropout), nn.Linear(dim * 4, dim))

    def forward(self, x):
        z = self.norm1(x)
        x = x + self.attention(z, z, z, need_weights=False)[0]
        return x + self.ff(self.norm2(x))


class XLSRDualGraph(SSLBase):
    def __init__(self, model_name, freeze=True, dim=128, dropout=0.2):
        super().__init__(model_name, freeze)
        self.projection = nn.Linear(self.hidden, dim)
        self.feature_projection = nn.Linear(8, dim)
        self.time_graph = nn.Sequential(AttentionBlock(dim), AttentionBlock(dim))
        self.feature_graph = nn.Sequential(AttentionBlock(dim), AttentionBlock(dim))
        self.head = nn.Sequential(nn.LayerNorm(dim * 4), nn.Dropout(dropout), nn.Linear(dim * 4, 256), nn.GELU(), nn.Linear(256, 5))

    def forward(self, audio):
        hidden = self.projection(self.features(audio))
        time_nodes = F.adaptive_avg_pool1d(hidden.transpose(1, 2), 64).transpose(1, 2)
        feature_nodes = self.feature_projection(F.adaptive_avg_pool1d(hidden.transpose(1, 2), 8))
        time_nodes = self.time_graph(time_nodes)
        feature_nodes = self.feature_graph(feature_nodes)
        pooled = torch.cat([time_nodes.mean(1), time_nodes.amax(1), feature_nodes.mean(1), feature_nodes.amax(1)], dim=-1)
        return self.head(pooled)


class TCMBlock(nn.Module):
    def __init__(self, dim, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.temporal = nn.Conv1d(dim, dim, 7, padding=3, groups=dim)
        self.gate = nn.Sequential(nn.Linear(dim, dim // 8), nn.SiLU(), nn.Linear(dim // 8, dim), nn.Sigmoid())
        self.attention = AttentionBlock(dim, dropout)

    def forward(self, x):
        z = self.norm(x)
        x = x + self.temporal(z.transpose(1, 2)).transpose(1, 2) * self.gate(z.mean(1)).unsqueeze(1)
        return self.attention(x)


class XLSRTCM(SSLBase):
    def __init__(self, model_name, freeze=True, dim=256, dropout=0.2):
        super().__init__(model_name, freeze)
        self.projection = nn.Linear(self.hidden, dim)
        self.tcm = nn.Sequential(TCMBlock(dim), TCMBlock(dim), TCMBlock(dim))
        self.pool = AttentivePool(dim)
        self.head = nn.Sequential(nn.LayerNorm(dim * 2), nn.Dropout(dropout), nn.Linear(dim * 2, 5))

    def forward(self, audio):
        return self.head(self.pool(self.tcm(self.projection(self.features(audio)))))


class XLSRSpectralFusion(SSLBase):
    def __init__(self, model_name, freeze=True, dropout=0.25):
        super().__init__(model_name, freeze)
        self.ssl_pool = AttentivePool(self.hidden)
        self.mels = nn.ModuleList([
            torchaudio.transforms.MelSpectrogram(sample_rate=CFG.sample_rate, n_fft=n_fft, hop_length=hop, n_mels=96, f_max=7600)
            for n_fft, hop in ((512, 160), (1024, 256), (2048, 512))
        ])
        self.spectral = nn.Sequential(
            nn.Conv2d(3, 32, 5, stride=2, padding=2), nn.BatchNorm2d(32), nn.GELU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.BatchNorm2d(64), nn.GELU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.GELU(), nn.AdaptiveAvgPool2d(1), nn.Flatten(),
        )
        self.head = nn.Sequential(nn.LayerNorm(self.hidden * 2 + 128), nn.Dropout(dropout), nn.Linear(self.hidden * 2 + 128, 256), nn.GELU(), nn.Linear(256, 5))

    def forward(self, audio):
        ssl_embedding = self.ssl_pool(self.features(audio))
        mel_features = []
        for transform in self.mels:
            feature = torch.log(transform(audio).clamp_min(1e-6))
            mel_features.append(F.interpolate(feature.unsqueeze(1), size=(96, 256), mode="bilinear", align_corners=False))
        spectral_embedding = self.spectral(torch.cat(mel_features, dim=1))
        return self.head(torch.cat([ssl_embedding, spectral_embedding], dim=-1))


## 12. 실험 설정

`exp03_aasist_rawboost`가 기본 학습·제출 모델입니다. 다른 모델은 `EXPERIMENTS_TO_RUN`에 추가합니다. SSL 모델은 Colab 시간과 평가 서버 60분 제한을 별도로 확인하세요.


In [ ]:
EXPERIMENTS = {
    "exp01_tiny": dict(model="tiny", batch=32, eval_batch=64, grad_accum=1, epochs=15, lr=3e-4, backbone_lr=3e-4, weight_decay=1e-4, patience=5, dropout=0.25),
    "exp02_aasist": dict(model="aasist", batch=16, eval_batch=32, grad_accum=1, epochs=20, lr=1e-4, backbone_lr=1e-4, weight_decay=1e-4, patience=6, dropout=0.2),
    "exp03_aasist_rawboost": dict(model="aasist", batch=16, eval_batch=32, grad_accum=1, epochs=24, lr=1e-4, backbone_lr=1e-4, weight_decay=1e-4, patience=7, rawboost_p=0.45, communication_p=0.30, dropout=0.2),
    "exp04_xlsr_pool": dict(model="xlsr_pool", batch=2, eval_batch=4, grad_accum=8, epochs=12, lr=1e-4, backbone_lr=5e-7, weight_decay=1e-4, patience=5, freeze_epochs=2, unfreeze_last=4, dropout=0.2),
    "exp05_xlsr_dualgraph": dict(model="xlsr_dualgraph", batch=2, eval_batch=4, grad_accum=8, epochs=14, lr=8e-5, backbone_lr=3e-7, weight_decay=1e-4, patience=5, freeze_epochs=3, unfreeze_last=4, dropout=0.2),
    "exp06_xlsr_tcm": dict(model="xlsr_tcm", batch=2, eval_batch=4, grad_accum=8, epochs=14, lr=8e-5, backbone_lr=3e-7, weight_decay=1e-4, patience=5, freeze_epochs=3, unfreeze_last=4, rawboost_p=0.20, communication_p=0.25, dropout=0.2),
    "exp07_multibranch": dict(model="multibranch", batch=2, eval_batch=4, grad_accum=8, epochs=14, lr=8e-5, backbone_lr=3e-7, weight_decay=1e-4, patience=5, freeze_epochs=3, unfreeze_last=4, communication_p=0.20, dropout=0.25),
}
display(pd.DataFrame(EXPERIMENTS).T)


## 13. 공통 학습 루프

Train worker는 epoch마다 새로 생성되어 `set_epoch()`이 실제 동적 믹싱에 반영됩니다. Validation/audit은 고정 recipe와 고정 crop을 사용합니다. Best checkpoint는 validation 공식 `Score`가 최대인 epoch입니다.


In [ ]:
RESUME_TRAINING = True
AUDIO_ERROR_LOG = []


def build_model(exp):
    if exp["model"] == "tiny":
        return TinyLogMel(exp.get("dropout", 0.25))
    if exp["model"] == "aasist":
        return official_aasist(False)
    if exp["model"] == "xlsr_pool":
        return XLSRPool(XLSR_MODEL, True, exp["dropout"])
    if exp["model"] == "xlsr_dualgraph":
        return XLSRDualGraph(XLSR_MODEL, True, dropout=exp["dropout"])
    if exp["model"] == "xlsr_tcm":
        return XLSRTCM(XLSR_MODEL, True, dropout=exp["dropout"])
    if exp["model"] == "multibranch":
        return XLSRSpectralFusion(XLSR_MODEL, True, exp["dropout"])
    raise KeyError(exp["model"])


def make_loaders(exp):
    train_dataset = DynamicMixDataset(
        recipe_by_split["train"], source_by_split["train"], training=True,
        rawboost_p=exp.get("rawboost_p", 0.0), communication_p=exp.get("communication_p", 0.0),
    )
    validation_dataset = DynamicMixDataset(recipe_by_split["validation"], source_by_split["validation"])
    train_loader = DataLoader(
        train_dataset, batch_size=exp["batch"], shuffle=True,
        num_workers=CFG.num_workers, pin_memory=DEVICE.type == "cuda",
        persistent_workers=False, drop_last=False,
    )
    validation_loader = DataLoader(
        validation_dataset, batch_size=exp["eval_batch"], shuffle=False,
        num_workers=CFG.num_workers, pin_memory=DEVICE.type == "cuda",
        persistent_workers=False,
    )
    return train_dataset, train_loader, validation_loader


def run_loader(model, loader, optimizer=None, scheduler=None, scaler=None, grad_accum=1, description="eval"):
    training = optimizer is not None
    model.train(training)
    if training:
        optimizer.zero_grad(set_to_none=True)
    total_loss = sample_count = skipped = 0
    all_targets, all_predictions, identifiers, recipe_types = [], [], [], []
    progress = tqdm(loader, total=len(loader), dynamic_ncols=True, leave=True, desc=description)
    for step, batch in enumerate(progress, 1):
        valid = batch["valid"].bool()
        skipped += int((~valid).sum())
        if not valid.any():
            continue
        audio = batch["audio"][valid].to(DEVICE, non_blocking=True)
        targets = batch["target"][valid].to(DEVICE, non_blocking=True)
        masks = batch["mask"][valid].to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
            logits = model(audio)
            unscaled_loss = masked_multitask_loss(logits, targets, masks)
            loss = unscaled_loss / (grad_accum if training else 1)
        if training:
            scaler.scale(loss).backward()
            if step % grad_accum == 0 or step == len(loader):
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
        count = int(targets.shape[0])
        total_loss += float(unscaled_loss.detach().cpu()) * count
        sample_count += count
        all_targets.append(targets.detach().cpu().numpy())
        all_predictions.append(torch.sigmoid(logits.detach()).cpu().numpy())
        flags = valid.tolist()
        identifiers.extend([value for value, keep in zip(batch["id"], flags) if keep])
        recipe_types.extend([value for value, keep in zip(batch["recipe_type"], flags) if keep])
        if step == 1 or step % 50 == 0 or step == len(loader):
            progress.set_postfix(loss=f"{total_loss / max(sample_count, 1):.4f}", samples=sample_count, skipped=skipped)
    if not all_targets:
        raise RuntimeError(f"{description}: valid audio가 없습니다.")
    targets = np.concatenate(all_targets)
    predictions = np.concatenate(all_predictions)
    report = report_from_arrays(targets, predictions)
    report.update(loss=total_loss / sample_count, samples=sample_count, skipped=skipped)
    return report, targets, predictions, identifiers, recipe_types


def fit_experiment(experiment_name):
    from transformers import get_cosine_schedule_with_warmup

    exp = dict(EXPERIMENTS[experiment_name])
    run_dir = RUN_ROOT / experiment_name
    run_dir.mkdir(parents=True, exist_ok=True)
    train_dataset, train_loader, validation_loader = make_loaders(exp)
    model = build_model(exp).to(DEVICE)
    backbone, head = [], []
    for name, parameter in model.named_parameters():
        (backbone if name.startswith("ssl.") else head).append(parameter)
    groups = []
    if backbone:
        groups.append({"params": backbone, "lr": exp["backbone_lr"]})
    if head:
        groups.append({"params": head, "lr": exp["lr"]})
    optimizer = torch.optim.AdamW(groups, weight_decay=exp["weight_decay"])
    updates_per_epoch = math.ceil(len(train_loader) / exp["grad_accum"])
    total_updates = max(1, updates_per_epoch * exp["epochs"])
    scheduler = get_cosine_schedule_with_warmup(optimizer, max(10, int(total_updates * 0.08)), total_updates)
    scaler = torch.cuda.amp.GradScaler(enabled=DEVICE.type == "cuda")

    best_score, stale, history, start_epoch = -float("inf"), 0, [], 1
    last_path = run_dir / "last.pt"
    if RESUME_TRAINING and last_path.exists():
        state = torch.load(last_path, map_location="cpu", weights_only=False)
        if state.get("config") == exp:
            model.load_state_dict(state["model_state"], strict=True)
            if state["epoch"] >= exp.get("freeze_epochs", 10**9) and hasattr(model, "unfreeze_last"):
                model.unfreeze_last(exp.get("unfreeze_last", 4))
            optimizer.load_state_dict(state["optimizer_state"])
            scheduler.load_state_dict(state["scheduler_state"])
            scaler.load_state_dict(state["scaler_state"])
            best_score, stale = float(state["best_score"]), int(state["stale"])
            history, start_epoch = list(state["history"]), int(state["epoch"]) + 1
            if state.get("finished"):
                return pd.DataFrame(history)

    for epoch in range(start_epoch, exp["epochs"] + 1):
        started = time.time()
        train_dataset.set_epoch(epoch)
        if epoch == exp.get("freeze_epochs", -1) + 1 and hasattr(model, "unfreeze_last"):
            model.unfreeze_last(exp.get("unfreeze_last", 4))
        train_report, *_ = run_loader(
            model, train_loader, optimizer, scheduler, scaler,
            exp["grad_accum"], f"{experiment_name} train {epoch}/{exp['epochs']}",
        )
        with torch.no_grad():
            validation_report, targets, predictions, ids, types = run_loader(
                model, validation_loader, description=f"{experiment_name} validation",
            )
        record = {
            "epoch": epoch, "train_loss": train_report["loss"],
            **{f"validation_{key}": value for key, value in validation_report.items()},
            "minutes": (time.time() - started) / 60,
        }
        history.append(record)
        pd.DataFrame(history).to_csv(run_dir / "history.csv", index=False)
        print(record)

        score = float(validation_report["score"])
        if score > best_score:
            best_score, stale = score, 0
            torch.save({
                "model_state": model.state_dict(), "experiment": experiment_name,
                "config": exp, "epoch": epoch, "validation_report": validation_report,
                "dacon_columns": DACON_PROBABILITY_COLUMNS,
                "source_manifest_sha256": hashlib.sha256(SOURCE_MANIFEST_PATH.read_bytes()).hexdigest(),
                "recipe_manifest_sha256": hashlib.sha256(RECIPE_MANIFEST_PATH.read_bytes()).hexdigest(),
                "repo_commits": repo_commits,
            }, run_dir / "best.pt")
            prediction_frame = pd.DataFrame({"ID": ids, "recipe_type": types})
            for index, column in enumerate(DACON_TRUTH_COLUMNS):
                prediction_frame[column] = targets[:, index]
            for index, column in enumerate(DACON_PROBABILITY_COLUMNS):
                prediction_frame[column] = predictions[:, index]
            prediction_frame.to_csv(run_dir / "validation_predictions.csv", index=False)
        else:
            stale += 1
        finished = stale >= exp["patience"] or epoch == exp["epochs"]
        torch.save({
            "model_state": model.state_dict(), "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(), "scaler_state": scaler.state_dict(),
            "config": exp, "epoch": epoch, "best_score": best_score,
            "stale": stale, "history": history, "finished": finished,
        }, last_path)
        if stale >= exp["patience"]:
            break
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return pd.DataFrame(history)


## 14. 학습 실행

기본값은 AASIST+RawBoost 하나입니다. Tiny smoke test를 먼저 하려면 목록 앞에 `exp01_tiny`를 추가하세요.


In [ ]:
RUN_TRAINING = True
EXPERIMENTS_TO_RUN = ["exp03_aasist_rawboost"]

if RUN_TRAINING:
    summaries = []
    for experiment_name in EXPERIMENTS_TO_RUN:
        history = fit_experiment(experiment_name)
        best_row = history.loc[history["validation_score"].idxmax()].to_dict()
        summaries.append({"experiment": experiment_name, **best_row})
        display(history)
    training_summary = pd.DataFrame(summaries).sort_values("validation_score", ascending=False)
    training_summary.to_csv(RUN_ROOT / "training_summary.csv", index=False)
    display(training_summary)


## 15. 고정 audit 2,500개 최종 평가

Validation으로 선택된 `best.pt`만 audit split에 적용합니다. Audit 결과를 보고 학습 설정을 되돌려 바꾸면 독립 평가가 아닙니다.


In [ ]:
RUN_AUDIT = True
AUDIT_EXPERIMENTS = EXPERIMENTS_TO_RUN

if RUN_AUDIT:
    audit_dataset = DynamicMixDataset(recipe_by_split["audit"], source_by_split["audit"])
    audit_rows = []
    for experiment_name in AUDIT_EXPERIMENTS:
        checkpoint_path = RUN_ROOT / experiment_name / "best.pt"
        checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
        exp = checkpoint["config"]
        model = build_model(exp)
        model.load_state_dict(checkpoint["model_state"], strict=True)
        model.to(DEVICE).eval()
        audit_loader = DataLoader(
            audit_dataset, batch_size=exp["eval_batch"], shuffle=False,
            num_workers=CFG.num_workers, pin_memory=DEVICE.type == "cuda",
            persistent_workers=False,
        )
        with torch.no_grad():
            report, targets, predictions, ids, types = run_loader(model, audit_loader, description=f"{experiment_name} audit")
        audit_rows.append({"experiment": experiment_name, **report})
        frame = pd.DataFrame({"ID": ids, "recipe_type": types})
        for index, column in enumerate(DACON_TRUTH_COLUMNS):
            frame[column] = targets[:, index]
        for index, column in enumerate(DACON_PROBABILITY_COLUMNS):
            frame[column] = predictions[:, index]
        frame.to_csv(RUN_ROOT / experiment_name / "audit_predictions.csv", index=False)
        del model
        gc.collect()
        torch.cuda.empty_cache()
    audit_summary = pd.DataFrame(audit_rows).sort_values("score", ascending=False)
    audit_summary.to_csv(RUN_ROOT / "audit_summary.csv", index=False)
    display(audit_summary)


## 16. DACON submit.zip 생성

기본 AASIST checkpoint와 필요한 공식 AASIST 소스/config를 `model/`에 포함합니다. 평가 서버에는 인터넷이 없으므로 inference 중 다운로드를 하지 않습니다.

```text
submit.zip
├── model/
│   ├── model.pt
│   └── aasist/
├── script.py
└── requirements.txt
```

`script.py`는 `data/test/`의 파일별로 독립 추론하고 `output/submission.csv`를 생성합니다.


In [ ]:
SUBMIT_EXPERIMENT = "exp03_aasist_rawboost"
BUILD_SUBMIT_ZIP = True
SUBMIT_ZIP_PATH = DRIVE_ROOT / "submit.zip"

INFERENCE_SCRIPT = r'''from __future__ import annotations

import json
import math
import subprocess
import sys
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torch.nn as nn
import torchaudio


ROOT = Path(__file__).resolve().parent
MODEL_DIR = ROOT / "model"
DATA_DIR = ROOT / "data"
TEST_DIR = DATA_DIR / "test"
OUTPUT_DIR = ROOT / "output"
OUTPUT_PATH = OUTPUT_DIR / "submission.csv"
SAMPLE_RATE = 16_000
PROBABILITY_COLUMNS = [
    "FILE_FAKE_PROB", "VOICE_FAKE_PROB", "MUSIC_FAKE_PROB",
    "VOICE_PRESENT_PROB", "MUSIC_PRESENT_PROB",
]
AUDIO_SUFFIXES = {".wav", ".mp3", ".flac", ".ogg", ".m4a", ".aac", ".amr"}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class TinyLogMel(nn.Module):
    def __init__(self, dropout=0.25):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=SAMPLE_RATE, n_fft=1024, win_length=400,
            hop_length=160, n_mels=96, f_min=20, f_max=7600,
        )
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.SiLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.SiLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.SiLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
        )
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(128, 5))

    def forward(self, audio):
        feature = torch.log(self.mel(audio).clamp_min(1e-6))
        feature = (feature - feature.mean((-2, -1), keepdim=True)) / (feature.std((-2, -1), keepdim=True) + 1e-5)
        return self.head(self.encoder(feature.unsqueeze(1)))


def build_aasist():
    repo = MODEL_DIR / "aasist"
    sys.path.insert(0, str(repo))
    with (repo / "config" / "AASIST.conf").open(encoding="utf-8") as file:
        model_config = json.load(file)["model_config"]
    from models.AASIST import Model as AASISTModel

    class Wrapper(nn.Module):
        def __init__(self):
            super().__init__()
            self.net = AASISTModel(model_config)
            self.net.out_layer = nn.Linear(self.net.out_layer.in_features, 5)

        def forward(self, audio):
            _, logits = self.net(audio, Freq_aug=False)
            return logits

    return Wrapper()


def load_model():
    checkpoint = torch.load(MODEL_DIR / "model.pt", map_location="cpu", weights_only=False)
    config = checkpoint["config"]
    if config["model"] == "tiny":
        model = TinyLogMel(config.get("dropout", 0.25))
    elif config["model"] == "aasist":
        model = build_aasist()
    else:
        raise RuntimeError(f"submit packager supports tiny/aasist, got {config['model']}")
    model.load_state_dict(checkpoint["model_state"], strict=True)
    model.to(DEVICE).eval()
    return model


def read_audio(path):
    try:
        audio, sample_rate = sf.read(path, dtype="float32", always_2d=True)
        waveform = torch.from_numpy(audio.mean(axis=1))
    except Exception:
        try:
            audio, sample_rate = librosa.load(path, sr=None, mono=True)
            waveform = torch.from_numpy(np.asarray(audio, dtype=np.float32))
        except Exception:
            decoded = subprocess.run(
                [
                    "ffmpeg", "-hide_banner", "-loglevel", "error", "-i", str(path),
                    "-ac", "1", "-ar", str(SAMPLE_RATE), "-f", "f32le", "pipe:1",
                ],
                stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True, timeout=120,
            )
            waveform = torch.from_numpy(np.frombuffer(decoded.stdout, dtype="<f4").copy())
            sample_rate = SAMPLE_RATE
    if waveform.numel() == 0:
        raise ValueError(f"empty audio: {path}")
    if sample_rate != SAMPLE_RATE:
        waveform = torchaudio.functional.resample(waveform, sample_rate, SAMPLE_RATE)
    return waveform.float().nan_to_num().clamp(-1, 1)


def make_segments(waveform, clip_samples=64_600):
    if waveform.numel() < clip_samples:
        waveform = waveform.repeat(math.ceil(clip_samples / waveform.numel()))
        return waveform[:clip_samples].unsqueeze(0)
    starts = list(range(0, waveform.numel() - clip_samples + 1, clip_samples))
    last = waveform.numel() - clip_samples
    if starts[-1] != last:
        starts.append(last)
    return torch.stack([waveform[start:start + clip_samples] for start in starts])


def predict_file(model, waveform, batch_size=16):
    segments = make_segments(waveform)
    outputs = []
    with torch.inference_mode():
        for start in range(0, len(segments), batch_size):
            logits = model(segments[start:start + batch_size].to(DEVICE))
            outputs.append(torch.sigmoid(logits).cpu().numpy())
    matrix = np.concatenate(outputs)
    fake_heads = 0.6 * matrix[:, :3].max(0) + 0.4 * matrix[:, :3].mean(0)
    presence_heads = 0.7 * matrix[:, 3:].max(0) + 0.3 * matrix[:, 3:].mean(0)
    file_direct, voice_fake, music_fake = fake_heads
    voice_present, music_present = presence_heads
    coherent_file = 1 - (1 - voice_fake * voice_present) * (1 - music_fake * music_present)
    file_fake = float(np.clip(0.7 * file_direct + 0.3 * coherent_file, 0, 1))
    return [
        file_fake, float(voice_fake), float(music_fake),
        float(voice_present), float(music_present),
    ]


def resolve_paths(sample):
    files = sorted(path for path in TEST_DIR.rglob("*") if path.is_file() and path.suffix.lower() in AUDIO_SUFFIXES)
    lookup = {}
    for path in files:
        lookup[path.stem] = path
        lookup[path.name] = path
    resolved = [lookup.get(str(identifier)) for identifier in sample["ID"]]
    if any(path is None for path in resolved):
        if len(files) != len(sample):
            missing = [str(identifier) for identifier, path in zip(sample["ID"], resolved) if path is None]
            raise FileNotFoundError(missing[:10])
        resolved = files
    return resolved


def main():
    sample = pd.read_csv(DATA_DIR / "sample_submission.csv", encoding="utf-8")
    required = ["ID", *PROBABILITY_COLUMNS]
    if list(sample.columns) != required:
        raise ValueError(f"unexpected columns: {list(sample.columns)}")
    model = load_model()
    rows = []
    for identifier, path in zip(sample["ID"].astype(str), resolve_paths(sample)):
        probabilities = predict_file(model, read_audio(path))
        rows.append([identifier, *probabilities])
    submission = pd.DataFrame(rows, columns=required)
    values = submission[PROBABILITY_COLUMNS].to_numpy(dtype=float)
    if len(submission) != len(sample):
        raise RuntimeError("row count mismatch")
    if not np.isfinite(values).all() or not ((values >= 0).all() and (values <= 1).all()):
        raise ValueError("probability outside [0, 1]")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    submission.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")
    print(f"saved {OUTPUT_PATH}: {len(submission)} rows")


if __name__ == "__main__":
    main()
'''

if BUILD_SUBMIT_ZIP:
    checkpoint_path = RUN_ROOT / SUBMIT_EXPERIMENT / "best.pt"
    if not checkpoint_path.exists():
        raise FileNotFoundError(f"먼저 학습을 완료하세요: {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    model_kind = checkpoint["config"]["model"]
    if model_kind not in {"tiny", "aasist"}:
        raise NotImplementedError("자동 패키징은 tiny/aasist checkpoint를 지원합니다.")

    stage = Path("/content/dacon_dynamic25k_submit")
    if stage.exists():
        shutil.rmtree(stage)
    model_dir = stage / "model"
    model_dir.mkdir(parents=True)
    shutil.copy2(checkpoint_path, model_dir / "model.pt")
    if model_kind == "aasist":
        shutil.copytree(
            REPO_ROOT / "aasist", model_dir / "aasist",
            ignore=shutil.ignore_patterns(".git", "__pycache__", "*.pyc", "LA", "database"),
        )
    (stage / "script.py").write_text(INFERENCE_SCRIPT, encoding="utf-8")
    (stage / "requirements.txt").write_text(
        "# DACON server preinstalls torch, torchaudio, pandas, numpy, librosa and soundfile.\n",
        encoding="utf-8",
    )
    import py_compile
    py_compile.compile(str(stage / "script.py"), doraise=True)

    with zipfile.ZipFile(SUBMIT_ZIP_PATH, "w", zipfile.ZIP_DEFLATED, allowZip64=True) as archive:
        archive.write(model_dir, "model/")
        for path in sorted(model_dir.rglob("*")):
            if path.is_file():
                archive.write(path, path.relative_to(stage).as_posix())
        archive.write(stage / "script.py", "script.py")
        archive.write(stage / "requirements.txt", "requirements.txt")
    with zipfile.ZipFile(SUBMIT_ZIP_PATH) as archive:
        names = archive.namelist()
        top_level = {name.split("/", 1)[0] for name in names}
        assert top_level == {"model", "script.py", "requirements.txt"}
        assert "model/model.pt" in names
        compressed_gb = SUBMIT_ZIP_PATH.stat().st_size / 1024**3
        uncompressed_gb = sum(info.file_size for info in archive.infolist()) / 1024**3
    if compressed_gb >= 10 or uncompressed_gb >= 32:
        raise RuntimeError({"compressed_gb": compressed_gb, "uncompressed_gb": uncompressed_gb})
    print("created:", SUBMIT_ZIP_PATH)
    print({"compressed_gb": compressed_gb, "uncompressed_gb": uncompressed_gb})


## 17. DACON dummy data 스모크 테스트

대회 `open.zip`의 `data/`를 `/content/dacon_open/data`에 두고 플래그를 켜면, 생성 zip을 새 폴더에 풀어 평가 서버와 같은 상대경로로 실행합니다.


In [ ]:
RUN_SUBMIT_SMOKE_TEST = False
DACON_DUMMY_DATA = Path("/content/dacon_open/data")

if RUN_SUBMIT_SMOKE_TEST:
    smoke_root = Path("/content/dacon_dynamic25k_smoke")
    if smoke_root.exists():
        shutil.rmtree(smoke_root)
    smoke_root.mkdir(parents=True)
    with zipfile.ZipFile(SUBMIT_ZIP_PATH) as archive:
        archive.extractall(smoke_root)
    shutil.copytree(DACON_DUMMY_DATA, smoke_root / "data")
    subprocess.run([sys.executable, "script.py"], cwd=smoke_root, check=True)
    result = pd.read_csv(smoke_root / "output" / "submission.csv")
    sample = pd.read_csv(smoke_root / "data" / "sample_submission.csv")
    assert list(result.columns) == ["ID", *DACON_PROBABILITY_COLUMNS]
    assert result["ID"].astype(str).tolist() == sample["ID"].astype(str).tolist()
    values = result[DACON_PROBABILITY_COLUMNS].to_numpy(dtype=float)
    assert np.isfinite(values).all() and ((0 <= values) & (values <= 1)).all()
    display(result)
    print("submit.zip smoke test passed")


## 18. 실행 체크리스트

- [ ] Kaggle voice dataset의 real/fake 각 6,250개 선택
- [ ] FMA medium(small+medium)에서 6,250개 선택, NoDerivatives 제외, license manifest 보존
- [ ] SONICS part_01/part_02 다운로드 후 metadata와 일치하는 가짜 곡 6,250개 선택
- [ ] 네 pool source manifest 총 25,000개
- [ ] pool별 train/validation/audit 5,000/625/625, source leakage 없음
- [ ] dynamic recipe 20,000/2,500/2,500, 총 25,000개
- [ ] FMA PANNs와 SONICS no_vocal을 DACON voice presence/fake 라벨에 반영
- [ ] SNR, overlap/partial/sequential, peak randomization, RawBoost/통신 열화 적용
- [ ] 5-head masked BCE와 공식 EER/ROC-AUC/ADS/CPS/Score 사용
- [ ] best.pt는 validation Score만으로 선택
- [ ] submit.zip 최상위가 model/, script.py, requirements.txt와 정확히 일치
- [ ] data/test/ → output/submission.csv, UTF-8, 5개 확률 [0,1]
- [ ] dummy 3개 스모크 테스트 통과 후 리더보드 제출
